<a href="https://colab.research.google.com/github/Gr1lledChee5e/LLM/blob/main/mas691_LLM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MAS 691 - Large Language Models


The purpose of this homework is to instll the concept of the Attention Block that forms the basis of modern LLM architectures, as well as train your own custom LLM under various parameter settings to study the impacts.

+ **Author:** Riley Yee


+ Homework 2
+ Due Wednesday, April 22th

# Part 1 - Attention Mechanisms

For Part 1, you will do one pass through the Attention process to predict the next word in a sequence. **We did exactly this in class**, so this should be straightforward. The only change is that I will ask you to do this using 3-dimensional embeddings rather than 2-dimensional embeddings.

In [ ]:
# Import any packages needed
import tiktoken
import requests
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

In [ ]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
# Sample vocab of 25 words
vocab_list = [
    (1, "analyze"),
    (2, "build"),
    (3, "code"),
    (4, "data"),
    (5, "discover"),
    (6, "things"),
    (7, "fast"),
    (8, "generate"),
    (9, "helpful"),
    (10, "intelligent"),
    (11, "learn"),
    (12, "that"),
    (13, "networks"),
    (14, "optimize"),
    (15, "process"),
    (16, "query"),
    (17, "result"),
    (18, "scalable"),
    (19, "train"),
    (20, "understand"),
    (21, "visualize"),
    (22, "computers"),
    (23, "people"),
    (24, "mechanism"),
    (25, "embedding")
]

**TASK:** Convert the token embeddings below to 3-dimensional embeddings (you can pick any numbers you like)

In [ ]:
# Token embeddings
embeddings = {
    1:  [0.50, 0.10, 0.15],
    2:  [0.60, 0.20, 0.25],
    3:  [0.70, 0.30, 0.35],
    4:  [0.80, 0.40, 0.45],
    5:  [0.90, 0.50, 0.55],
    6:  [1.00, 0.60, 0.65],
    7:  [1.10, 0.70, 0.75],
    8:  [1.20, 0.80, 0.85],
    9:  [1.30, 0.90, 0.95],
    10: [1.40, 1.00, 1.05],
    11: [1.50, 1.10, 1.15],
    12: [1.60, 1.20, 1.25],
    13: [1.70, 1.30, 1.35],
    14: [1.80, 1.40, 1.45],
    15: [1.90, 1.50, 1.55],
    16: [2.00, 1.60, 1.65],
    17: [2.10, 1.70, 1.75],
    18: [2.20, 1.80, 1.85],
    19: [2.30, 1.90, 1.95],
    20: [2.40, 2.00, 2.05],
    21: [2.50, 2.10, 2.15],
    22: [2.60, 2.20, 2.25],
    23: [2.70, 2.30, 2.35],
    24: [2.80, 2.40, 2.45],
    25: [2.90, 2.50, 2.55]
}

In [ ]:
# Input sequence of at least 5 words
#   I tried to throw some words in there so you could form sentences, but ultimately any sequence of 5 tokens is fine.
input_sequence = 'intelligent networks learn that data'

In [ ]:
# Tokenize
tokenizer = tiktoken.encoding_for_model('gpt-2')
tokenizer_gpt3 = tiktoken.encoding_for_model('text-davinci-003')
tokenizer_gpt4 = tiktoken.encoding_for_model('gpt-4')

In [ ]:
word_to_id = {word: idx for idx, word in vocab_list}
tokens = [word_to_id[w] for w in input_sequence.split()]
print(f'Tokens: {tokens}')  # [10, 13, 11, 12, 4]

Tokens: [10, 13, 11, 12, 4]


In [ ]:
W_q = np.identity(2)
W_k = np.identity(2)
W_v = np.identity(2)
print('Wq:\n', W_q)
print('Wk:\n', W_k)
print('Wv:\n', W_v)

Wq:
 [[1. 0.]
 [0. 1.]]
Wk:
 [[1. 0.]
 [0. 1.]]
Wv:
 [[1. 0.]
 [0. 1.]]


In [ ]:
# Define the X embedding matrix using your token embeddings above
# Helper to map words from vocab_list to their custom IDs
word_to_id_map = {word: id for id, word in vocab_list}

# Tokenize the input sequence using the custom vocab_list
# This assumes all words in input_sequence are present in vocab_list
input_tokens_ids = [word_to_id_map[word] for word in input_sequence.split()]

X = np.array([embeddings[t_id] for t_id in input_tokens_ids])
print('X:\n', X)

X:
 [[1.4  1.   1.05]
 [1.7  1.3  1.35]
 [1.5  1.1  1.15]
 [1.6  1.2  1.25]
 [0.8  0.4  0.45]]


In [ ]:
# Initialize weight matrices W_q, W_k, and W_v
#    Use random numbers, negatives are ok too (consider numbers around the same magnitude of the embeddings above)
#    Since your embeddings now have 3 dimensions, these should also be 3x3 matrices rather than the 2x2's we had in class
W_q = np.identity(2)
W_k = np.identity(2)
W_v = np.identity(2)
print('Wq:\n', W_q)
print('Wk:\n', W_k)
print('Wv:\n', W_v)

Wq:
 [[1. 0.]
 [0. 1.]]
Wk:
 [[1. 0.]
 [0. 1.]]
Wv:
 [[1. 0.]
 [0. 1.]]


In [ ]:
np.random.seed(42)
W_q = np.array([[ 0.10, -0.20,  0.30],
                [ 0.40,  0.10, -0.10],
                [-0.20,  0.30,  0.20]])

W_k = np.array([[ 0.20,  0.10, -0.30],
                [-0.10,  0.40,  0.20],
                [ 0.30, -0.20,  0.10]])

W_v = np.array([[ 0.30,  0.20,  0.10],
                [ 0.10, -0.10,  0.40],
                [-0.30,  0.20,  0.30]])

print('W_q:\n', W_q)
print('W_k:\n', W_k)
print('W_v:\n', W_v)

W_q:
 [[ 0.1 -0.2  0.3]
 [ 0.4  0.1 -0.1]
 [-0.2  0.3  0.2]]
W_k:
 [[ 0.2  0.1 -0.3]
 [-0.1  0.4  0.2]
 [ 0.3 -0.2  0.1]]
W_v:
 [[ 0.3  0.2  0.1]
 [ 0.1 -0.1  0.4]
 [-0.3  0.2  0.3]]


In [ ]:
def softmax(x):
    e_x = np.exp(x)
    return e_x / e_x.sum(axis=1, keepdims=True)

Q = np.dot(X, W_q)
K = np.dot(X, W_k)
V = np.dot(X, W_v)
print('Q:\n', Q)
print('K:\n', K)
print('V:\n', V)

Q:
 [[0.33  0.135 0.53 ]
 [0.42  0.195 0.65 ]
 [0.36  0.155 0.57 ]
 [0.39  0.175 0.61 ]
 [0.15  0.015 0.29 ]]
K:
 [[ 0.495  0.33  -0.115]
 [ 0.615  0.42  -0.115]
 [ 0.535  0.36  -0.115]
 [ 0.575  0.39  -0.115]
 [ 0.255  0.15  -0.115]]
V:
 [[0.205 0.39  0.855]
 [0.235 0.48  1.095]
 [0.215 0.42  0.935]
 [0.225 0.45  1.015]
 [0.145 0.21  0.375]]


In [ ]:
# Helper to map words from vocab_list to their custom IDs
word_to_id_map = {word: id for id, word in vocab_list}

# Tokenize the input sequence using the custom vocab_list
# This assumes all words in input_sequence are present in vocab_list
input_tokens_ids = [word_to_id_map[word] for word in input_sequence.split()]

X = np.array([embeddings[t_id] for t_id in input_tokens_ids])
print(f'X (input embedding matrix):\n{X}')

X (input embedding matrix):
[[1.4  1.   1.05]
 [1.7  1.3  1.35]
 [1.5  1.1  1.15]
 [1.6  1.2  1.25]
 [0.8  0.4  0.45]]


In [ ]:
Q = np.dot(X, W_q)
K = np.dot(X, W_k)
V = np.dot(X, W_v)
print('Q:\n', Q)
print('K:\n', K)
print('V:\n', V)

Q:
 [[0.33  0.135 0.53 ]
 [0.42  0.195 0.65 ]
 [0.36  0.155 0.57 ]
 [0.39  0.175 0.61 ]
 [0.15  0.015 0.29 ]]
K:
 [[ 0.495  0.33  -0.115]
 [ 0.615  0.42  -0.115]
 [ 0.535  0.36  -0.115]
 [ 0.575  0.39  -0.115]
 [ 0.255  0.15  -0.115]]
V:
 [[0.205 0.39  0.855]
 [0.235 0.48  1.095]
 [0.215 0.42  0.935]
 [0.225 0.45  1.015]
 [0.145 0.21  0.375]]


In [ ]:
# Compute the matrix of attention scores (don't forget the denominator - sqrt(??))
As = np.dot(Q, K.T) / np.sqrt(2)
print('Attention Scores:\n', As)

Attention Scores:
 [[0.10390934 0.14050212 0.11610693 0.12830453 0.03072379]
 [0.13965359 0.1877015  0.15566956 0.17168553 0.04355778]
 [0.11582409 0.15623524 0.12929447 0.14276486 0.03500179]
 [0.12773884 0.17196837 0.14248202 0.15722519 0.03927978]
 [0.03242085 0.04610336 0.03698168 0.04154252 0.00505581]]


In [ ]:
# Compute the matrix of attention weights
def softmax(x):
    e_x = np.exp(x)
    return e_x / e_x.sum(axis=1, keepdims=True)

Aw = softmax(As)
print('Attention Weights:\n', Aw)

Attention Weights:
 [[0.19985345 0.2073021  0.20230611 0.20478887 0.18574947]
 [0.19974858 0.20958039 0.20297351 0.2062505  0.18144702]
 [0.19982157 0.20806197 0.20253145 0.20527808 0.18430693]
 [0.1997866  0.2088214  0.20275391 0.20576529 0.18287279]
 [0.19997931 0.20273434 0.20089347 0.20181181 0.19458107]]


In [ ]:
# Compute the final output embeddings
Z = np.dot(Aw, V)
print('Final Output Embeddings:\n', Z)

Final Output Embeddings:
 [[0.20619293 0.3935788  0.86454347]
 [0.20655534 0.39466601 0.86744269]
 [0.20631432 0.39394296 0.86551455]
 [0.20643512 0.39430536 0.86648096]
 [0.20545234 0.39135701 0.85861869]]


In [ ]:
# Calculate the cosine similarity between the last output embedding and the token embeddings above to determine the next token
last_output = Z[-1].reshape(1, -1)

print('Cosine similarity to each vocab word:')
similarities = []
for token_id, word in vocab_list:
    emb = np.array(embeddings[token_id]).reshape(1, -1)
    sim = cosine_similarity(last_output, emb)[0][0]
    similarities.append((sim, token_id, word))
    print(f'  {word:<12} (id={token_id:2d}): {sim:.6f}')

best = max(similarities)
print(f'\nPredicted next token: "{best[2]}" (id={best[1]})')
print(f'\nCompleted Sentence: {input_sequence} {best[2]}')

Cosine similarity to each vocab word:
  analyze      (id= 1): 0.527302
  build        (id= 2): 0.633719
  code         (id= 3): 0.694009
  data         (id= 4): 0.731483
  discover     (id= 5): 0.756563
  things       (id= 6): 0.774335
  fast         (id= 7): 0.787501
  generate     (id= 8): 0.797602
  helpful      (id= 9): 0.805573
  intelligent  (id=10): 0.812011
  learn        (id=11): 0.817311
  that         (id=12): 0.821745
  networks     (id=13): 0.825506
  optimize     (id=14): 0.828735
  process      (id=15): 0.831536
  query        (id=16): 0.833987
  result       (id=17): 0.836149
  scalable     (id=18): 0.838071
  train        (id=19): 0.839789
  understand   (id=20): 0.841334
  visualize    (id=21): 0.842731
  computers    (id=22): 0.844000
  people       (id=23): 0.845158
  mechanism    (id=24): 0.846219
  embedding    (id=25): 0.847193

Predicted next token: "embedding" (id=25)

Completed Sentence: intelligent networks learn that data embedding


**FILL THIS IN:**

**Completed Sentence:** Your Input Sequence + Next Word Predicted Above

# Part 2 - Training a Custom LLM

In [ ]:
import torch
import math
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import tiktoken
tokenizer = tiktoken.encoding_for_model('gpt-2')

### Part 2a - Text Data

You will need text data to train your LLM. You can either:
    
1. Find some text online, copy/paste it into a txt file (remove formatting, clean it up as needed, etc)
2. Go to ChatGPT and ask it to write you a 2,500 word story (or give you 2,500 words on a topic that interests you)

Either way, please put together a .txt file with approximately 2,500 words. You can look at the um-mission-statement.txt file as an example

In [ ]:
# Read in your text data here (please upload your .txt file with your HW submission)
raw_text = """Artificial intelligence is reshaping the way businesses approach marketing. From targeted advertising to automated email campaigns and search engine optimization, AI-driven systems are enabling brands to connect with their audiences more precisely, efficiently, and profitably than ever before. The transformation is not incremental; it represents a fundamental shift in how marketing decisions are made, how campaigns are executed, and how results are measured. Businesses that fail to adopt AI-powered marketing tools are already falling behind competitors who have made machine intelligence central to their growth strategies. This shift is not simply about adopting new tools, but about rethinking the entire marketing function as a data-driven, continuously learning system that evolves over time.

The modern marketing landscape is defined by data. Every click, scroll, purchase, and abandoned cart generates a signal that reflects some aspect of consumer intent or behavior. Historically, marketers lacked the computational capacity to process and act on this volume of information in real time. Decisions were based on aggregated reports, intuition, and limited testing frameworks. AI changes that equation entirely. Machine learning models can ingest millions of behavioral data points simultaneously and identify patterns that no human analyst could detect, even with extensive experience. These patterns power recommendation engines, automated ad bidding systems, and content personalization tools that now drive billions of dollars in revenue across industries worldwide.

AI systems in marketing operate across three primary domains: paid advertising, email marketing, and search engine optimization. Each of these domains has been fundamentally transformed by machine intelligence, and together they form an interconnected ecosystem that modern marketers must understand and manage. Beyond these core areas, AI is also influencing content creation, customer segmentation, predictive analytics, conversion rate optimization, and competitive intelligence. The breadth of this transformation means that marketing teams today must combine creative thinking with technical literacy. The marketer of the present and future is not just a communicator, but also a systems thinker who understands how algorithms influence outcomes and how data informs strategic decisions.

Paid advertising has undergone one of the most dramatic transformations due to AI. In its early stages, online advertising functioned as a relatively simple auction system where advertisers bid on keywords, and the highest bidder secured the top placement. Today, platforms such as Google and Meta use sophisticated machine learning models to evaluate the probability that a user will engage with or convert from a given advertisement. This predicted probability is combined with bid price and other quality signals to determine which ad is shown. As a result, relevance has become more important than raw spending power. A highly relevant ad with a lower bid can outperform a higher bid if the system predicts stronger user engagement and conversion likelihood.

These AI-driven advertising systems are continuously learning. Every impression, click, and conversion provides feedback that refines the model. Over time, this allows the system to make increasingly accurate predictions about which combinations of creative elements, audience segments, and placements will produce the best results. Marketers who understand this feedback loop can design campaigns that accelerate learning. For example, consolidating campaigns, maintaining consistent creative during testing periods, and allocating sufficient budget during early stages all help the algorithm gather meaningful data faster. This reduces inefficiencies and allows campaigns to reach optimal performance more quickly while minimizing wasted ad spend.

Audience targeting has also been significantly enhanced by AI. Traditional segmentation methods relied heavily on demographic data such as age, gender, and location. While still relevant, these variables are no longer sufficient on their own. Machine learning models now build complex behavioral profiles based on browsing patterns, purchase history, device usage, and interaction with digital content. Lookalike modeling enables platforms to identify new users who resemble a business’s most valuable customers across dozens or even hundreds of dimensions. This allows marketers to expand their reach while maintaining high levels of relevance, which directly improves campaign performance and return on investment.

Creative optimization represents another major advancement. Dynamic creative optimization systems automatically generate and test combinations of headlines, images, and calls to action. Instead of manually testing a small number of variations, marketers can upload large sets of creative assets and allow the system to determine which combinations perform best. This process occurs in real time, with the system continuously shifting impressions toward higher-performing variants. Interestingly, the combinations that perform best are often not those that marketers would have predicted, highlighting the value of data-driven experimentation over intuition alone and reinforcing the importance of trusting algorithmic insights.

Predictive bidding is one of the most technically sophisticated applications of AI in advertising. Rather than setting static bids, automated systems evaluate each individual auction in real time, considering contextual factors such as user behavior, location, device type, and time of day. The system then determines the optimal bid to maximize conversions or revenue within a given budget constraint. This level of precision would be impossible to achieve manually, particularly at scale. As a result, AI-driven bidding strategies consistently outperform traditional approaches, provided that sufficient historical data is available to train the models effectively and produce reliable predictions.

Email marketing, despite being one of the oldest digital marketing channels, remains one of the most effective, and AI has significantly enhanced its capabilities. The core challenge in email marketing has always been delivering relevant content to diverse audiences at scale. Sending the same message to an entire subscriber list often results in low engagement and increased unsubscribe rates. AI enables a more personalized approach by tailoring content, timing, and frequency to individual users based on their unique behaviors, preferences, and historical interactions with the brand.

Send time optimization is a widely used feature that illustrates this shift. Instead of sending emails at a fixed time, AI systems analyze historical engagement data to determine when each subscriber is most likely to open an email. This ensures that messages are delivered at optimal times for each individual, increasing the likelihood of engagement without requiring additional effort from marketers and improving overall campaign performance metrics.

Subject line optimization is another area where AI provides significant value. By analyzing large datasets of past campaigns, machine learning models can predict how different subject lines will perform. This allows marketers to test variations and select the most effective options before sending emails. Factors such as tone, length, personalization, and urgency are evaluated in combination, enabling more accurate predictions than traditional A/B testing alone and helping marketers refine their messaging strategies more effectively.

Content personalization goes even further by dynamically adjusting the content of each email based on user behavior. This can include product recommendations, personalized offers, and tailored messaging that reflects the user’s position in the customer journey. For example, a returning customer may receive loyalty rewards, while a new subscriber receives introductory offers. This level of personalization enhances the customer experience, increases engagement, and drives higher conversion rates by delivering content that is directly relevant to each recipient.

Churn prediction is another powerful application of AI in email marketing. By analyzing patterns in user behavior, such as declining engagement or reduced purchase frequency, models can identify customers who are at risk of disengaging. This allows businesses to intervene with targeted retention campaigns before losing the customer entirely. These proactive strategies are far more effective than reactive approaches, as they address issues before they become irreversible and help maintain long-term customer relationships.

Search engine optimization has also evolved significantly with the integration of AI. Early SEO strategies focused primarily on keyword matching and backlink acquisition. Modern search engines, however, use machine learning to understand user intent, context, and content quality. This means that effective SEO now requires a deeper understanding of topics, user needs, and content structure rather than relying solely on keyword density or simple ranking tactics.

AI tools have become essential for keyword research and content strategy. These tools can identify clusters of related keywords, analyze competitor strategies, and uncover gaps in existing content. This allows marketers to develop comprehensive content strategies that address user intent more effectively. Additionally, AI can assist in content creation by generating outlines, suggesting improvements, and optimizing for readability and relevance, which significantly reduces the time required to produce high-quality content.

The most advanced marketing organizations recognize that these channels do not operate in isolation. Instead, they function as components of an integrated system powered by shared data and coordinated AI models. A user’s interaction with one channel influences their experience across others. For example, a user who visits a website through organic search may later be targeted with personalized ads, followed by tailored email campaigns. Each interaction generates data that feeds back into the system, improving future performance and creating a continuous cycle of optimization.

Large language models have further expanded the capabilities of AI in marketing. These models can generate content, analyze data, and interact with customers in natural language. They enable faster campaign development, more efficient customer support, and deeper insights into performance metrics. As these technologies continue to evolve, their impact on marketing will only increase, shaping the way businesses communicate and engage with their audiences.

In conclusion, artificial intelligence is fundamentally transforming marketing by enabling more precise targeting, personalized experiences, and data-driven decision-making. Businesses that embrace these technologies are gaining a significant competitive advantage, while those that resist change risk becoming obsolete. The future of marketing will be defined by the ability to integrate AI into every aspect of strategy and execution, creating systems that learn, adapt, and improve continuously over time.
"""

print(f'Word count: {len(raw_text.split())}')

Word count: 1570


###Part 2


In this portion you are going to train a handful of models, tweaking one parameter each time to see how it affects performance.

Before you jump in, ask your favorite LLM what you can expect. Tell it you are training an LLM and are considering adjusting various parameter settings. Ask it to describe the impact on performance for adjusting:

1. Learning rate
2. Context length
3. Embedding dimension
4. Number of attention heads
5. Number of transformer blocks

**Summarize what you learned from its response. Which do expect to have the biggest impact? The least impact? Which settings would you expect to give you the best result?**



**TASK:**

In the code blocks below, please find the relevant components from the *Day 6 - Model for Training* file and paste them in. I listed them in separate blocks so you can avoid any unnecessary or redundant code I had in there, and so you can *hopefully* get an idea of what each component is there for.

You are going to train 4 models. Choose any two of the issues above (Learning rate, context length, embedding dimension, number of attention heads, number of transformer blocks) and adjust that parameter in the GPT_CONFIG_124M dictionary provided (leave everything else as is, also note that the learning rate is specified in the optimizer code when you kick off the training as lr=0.0004 in the code I gave you).

1. Learning Rate: lr = 0.0004 vs 0.004
2. Context Length: context_length = 36 vs 124
3. Embedding Dimension: emb_dim = 384 vs 768
4. Attention Heads: n_heads = 2 vs 12
5. Transformer Blocks: n_layers = 2 vs 12

For example, if you pick context length and transformer blocks (similar for any of the others), you will fit 4 models:

+ **Model 1:** Adjust GPT_CONFIG_124M to set context_length to 36
+ **Model 2:** Adjust GPT_CONFIG_124M to set context_length to 124
+ **Model 3:** Adjust GPT_CONFIG_124M to set n_layers to 2
+ **Model 4:** Adjust GPT_CONFIG_124M to set n_layers to 12

Everything else can remain as provided in the code below.

In [ ]:
class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []
        token_ids = tokenizer.encode(txt)
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i+max_length]
            target_chunk = token_ids[i+1:i+max_length+1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

In [ ]:
def create_dataloader_v1(txt, batch_size=4, max_length=124, stride=124, shuffle=True, drop_last=True, num_workers=0):
    tokenizer = tiktoken.get_encoding('gpt2')
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, drop_last=drop_last, num_workers=num_workers)
    return dataloader

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert (d_out % num_heads == 0), "d_out must be divisible by num_heads"
        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer('mask', torch.triu(torch.ones(context_length, context_length), diagonal=1))

    def forward(self, x):
        b, num_tokens, d_in = x.shape
        keys    = self.W_key(x)
        queries = self.W_query(x)
        values  = self.W_value(x)
        keys    = keys.view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)
        values  = values.view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)
        attn_scores = queries @ keys.transpose(2, 3)
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]
        attn_scores.masked_fill_(mask_bool, -torch.inf)
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)
        context_vec = (attn_weights @ values).transpose(1, 2)
        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
        return self.out_proj(context_vec)

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert (d_out % num_heads == 0), "d_out must be divisible by num_heads"
        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer('mask', torch.triu(torch.ones(context_length, context_length), diagonal=1))

    def forward(self, x):
        b, num_tokens, d_in = x.shape
        keys    = self.W_key(x)
        queries = self.W_query(x)
        values  = self.W_value(x)
        keys    = keys.view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)
        values  = values.view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)
        attn_scores = queries @ keys.transpose(2, 3)
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]
        attn_scores.masked_fill_(mask_bool, -torch.inf)
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)
        context_vec = (attn_weights @ values).transpose(1, 2)
        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
        return self.out_proj(context_vec)

In [ ]:
class LayerNorm(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var  = x.var(dim=-1, keepdim=True, unbiased=False)
        norm_x = (x - mean) / torch.sqrt(var + self.eps)
        return self.scale * norm_x + self.shift


class GELU(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        return 0.5 * x * (1 + torch.tanh(
            torch.sqrt(torch.tensor(2.0 / torch.pi)) * (x + 0.044715 * torch.pow(x, 3))
        ))


class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(cfg['emb_dim'], 4 * cfg['emb_dim']),
            GELU(),
            nn.Linear(4 * cfg['emb_dim'], cfg['emb_dim'])
        )

    def forward(self, x):
        return self.layers(x)

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att = MultiHeadAttention(
            d_in=cfg['emb_dim'], d_out=cfg['emb_dim'],
            context_length=cfg['context_length'], num_heads=cfg['n_heads'],
            dropout=cfg['drop_rate'], qkv_bias=cfg['qkv_bias']
        )
        self.ff = FeedForward(cfg)
        self.norm1 = LayerNorm(cfg['emb_dim'])
        self.norm2 = LayerNorm(cfg['emb_dim'])
        self.drop_shortcut = nn.Dropout(cfg['drop_rate'])

    def forward(self, x):
        shortcut = x
        x = self.norm1(x)
        x = self.att(x)
        x = self.drop_shortcut(x)
        x = x + shortcut
        shortcut = x
        x = self.norm2(x)
        x = self.ff(x)
        x = self.drop_shortcut(x)
        x = x + shortcut
        return x

In [ ]:
class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb  = nn.Embedding(cfg['vocab_size'], cfg['emb_dim'])
        self.pos_emb  = nn.Embedding(cfg['context_length'], cfg['emb_dim'])
        self.drop_emb = nn.Dropout(cfg['drop_rate'])
        self.trf_blocks = nn.Sequential(*[TransformerBlock(cfg) for _ in range(cfg['n_layers'])])
        self.final_norm = LayerNorm(cfg['emb_dim'])
        self.out_head   = nn.Linear(cfg['emb_dim'], cfg['vocab_size'], bias=False)

    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape
        tok_embeds = self.tok_emb(in_idx)
        pos_embeds = self.pos_emb(torch.arange(seq_len, device=in_idx.device))
        x = tok_embeds + pos_embeds
        x = self.drop_emb(x)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        return self.out_head(x)

In [ ]:
def generate_text_simple(model, idx, max_new_tokens, context_size):
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -context_size:]
        with torch.no_grad():
            logits = model(idx_cond)
        logits = logits[:, -1, :]
        idx_next = torch.argmax(torch.softmax(logits, dim=-1), dim=-1, keepdim=True)
        idx = torch.cat((idx, idx_next), dim=1)
    return idx

def text_to_token_ids(text, tokenizer):
    encoded = tokenizer.encode(text, allowed_special={'<|endoftext|>'})
    return torch.tensor(encoded).unsqueeze(0)

def token_ids_to_text(token_ids, tokenizer):
    return tokenizer.decode(token_ids.squeeze(0).tolist())

def calc_loss_batch(input_batch, target_batch, model, device):
    input_batch  = input_batch.to(device)
    target_batch = target_batch.to(device)
    logits = model(input_batch)
    return torch.nn.functional.cross_entropy(logits.flatten(0, 1), target_batch.flatten())

def calc_loss_loader(data_loader, model, device, num_batches=None):
    total_loss = 0.
    if len(data_loader) == 0:
        return float('nan')
    num_batches = len(data_loader) if num_batches is None else min(num_batches, len(data_loader))
    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i < num_batches:
            total_loss += calc_loss_batch(input_batch, target_batch, model, device).item()
        else:
            break
    return total_loss / num_batches

def evaluate_model(model, train_loader, val_loader, device, eval_iter):
    model.eval()
    with torch.no_grad():
        train_loss = calc_loss_loader(train_loader, model, device, num_batches=eval_iter)
        val_loss   = calc_loss_loader(val_loader,   model, device, num_batches=eval_iter)
    model.train()
    return train_loss, val_loss

def generate_and_print_sample(model, tokenizer, device, start_context):
    model.eval()
    context_size = model.pos_emb.weight.shape[0]
    encoded = text_to_token_ids(start_context, tokenizer).to(device)
    with torch.no_grad():
        token_ids = generate_text_simple(model=model, idx=encoded, max_new_tokens=50, context_size=context_size)
    print(token_ids_to_text(token_ids, tokenizer).replace('\n', ' '))
    model.train()

def train_model_simple(model, train_loader, val_loader, optimizer, device,
                       num_epochs, eval_freq, eval_iter, start_context, tokenizer):
    train_losses, val_losses, track_tokens_seen = [], [], []
    tokens_seen, global_step = 0, -1
    for epoch in range(num_epochs):
        model.train()
        for input_batch, target_batch in train_loader:
            optimizer.zero_grad()
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            loss.backward()
            optimizer.step()
            tokens_seen += input_batch.numel()
            global_step += 1
            if global_step % eval_freq == 0:
                train_loss, val_loss = evaluate_model(model, train_loader, val_loader, device, eval_iter)
                train_losses.append(train_loss)
                val_losses.append(val_loss)
                track_tokens_seen.append(tokens_seen)
                print(f"  Step {global_step:4d} | Train loss {train_loss:.3f} | Val loss {val_loss:.3f}")
        print(f"Epoch {epoch+1} sample:")
        generate_and_print_sample(model, tokenizer, device, start_context)
        print()
    return train_losses, val_losses, track_tokens_seen

In [ ]:
train_ratio = 0.90
split_idx   = int(train_ratio * len(raw_text))
train_data  = raw_text[:split_idx]
val_data    = raw_text[split_idx:]

In [ ]:
# Model 1 — emb_dim=384
CFG1 = {'vocab_size':50257,'context_length':124,'emb_dim':384,'n_heads':12,'n_layers':12,'drop_rate':0.1,'qkv_bias':False}
torch.manual_seed(123)
model1 = GPTModel(CFG1)

# Model 2 — emb_dim=768
CFG2 = {'vocab_size':50257,'context_length':124,'emb_dim':768,'n_heads':12,'n_layers':12,'drop_rate':0.1,'qkv_bias':False}
torch.manual_seed(123)
model2 = GPTModel(CFG2)

# Model 3 — n_layers=2
CFG3 = {'vocab_size':50257,'context_length':124,'emb_dim':768,'n_heads':12,'n_layers':2,'drop_rate':0.1,'qkv_bias':False}
torch.manual_seed(123)
model3 = GPTModel(CFG3)

# Model 4 — n_layers=12
CFG4 = {'vocab_size':50257,'context_length':124,'emb_dim':768,'n_heads':12,'n_layers':12,'drop_rate':0.1,'qkv_bias':False}
torch.manual_seed(123)
model4 = GPTModel(CFG4)

In [ ]:
#train mdoel 1
device = 'cpu'
start_context = 'AI marketing systems can'

print('=' * 60)
print('MODEL 1 — emb_dim=384')
print('=' * 60)
torch.manual_seed(123)
train_loader1 = create_dataloader_v1(train_data, batch_size=2, max_length=124, stride=124, shuffle=True,  drop_last=True)
val_loader1   = create_dataloader_v1(val_data,   batch_size=2, max_length=124, stride=124, shuffle=False, drop_last=False)
model1.to(device)
optimizer1 = torch.optim.AdamW(model1.parameters(), lr=0.0004, weight_decay=0.1)
train_losses1, val_losses1, _ = train_model_simple(
    model1, train_loader1, val_loader1, optimizer1, device,
    num_epochs=10, eval_freq=5, eval_iter=5, start_context=start_context, tokenizer=tokenizer
)
print(f'Final Train Loss: {train_losses1[-1]:.3f}')
print(f'Final Val   Loss: {val_losses1[-1]:.3f}')
print(f'Val Perplexity:   {math.exp(val_losses1[-1]):.2f}')

MODEL 1 — emb_dim=384
  Step    0 | Train loss 10.303 | Val loss 10.397
  Step    5 | Train loss 9.090 | Val loss 9.173
Epoch 1 sample:
AI marketing systems can,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,

  Step   10 | Train loss 8.161 | Val loss 8.434
Epoch 2 sample:
AI marketing systems can, and, and, and, and, and, and, and, and, and, and, and, and, and, and, and, and, and, and, and, and, and, and, and, and, and

  Step   15 | Train loss 7.096 | Val loss 7.741
Epoch 3 sample:
AI marketing systems can, and, and, and, and, and, and, and, and, and, and, and, and, and, and, and, and, and, and, and, and, and, and and, and, and,

  Step   20 | Train loss 5.941 | Val loss 7.349
Epoch 4 sample:
AI marketing systems can.                                                 

  Step   25 | Train loss 5.181 | Val loss 7.105
Epoch 5 sample:
AI marketing systems can. This allows the and. This. This allows. This allows the, and and, and and and and content, and, and.                      

  St

In [ ]:
#train model 2
print('=' * 60)
print('MODEL 2 — emb_dim=768')
print('=' * 60)
torch.manual_seed(123)
train_loader2 = create_dataloader_v1(train_data, batch_size=2, max_length=124, stride=124, shuffle=True,  drop_last=True)
val_loader2   = create_dataloader_v1(val_data,   batch_size=2, max_length=124, stride=124, shuffle=False, drop_last=False)
model2.to(device)
optimizer2 = torch.optim.AdamW(model2.parameters(), lr=0.0004, weight_decay=0.1)
train_losses2, val_losses2, _ = train_model_simple(
    model2, train_loader2, val_loader2, optimizer2, device,
    num_epochs=10, eval_freq=5, eval_iter=5, start_context=start_context, tokenizer=tokenizer
)
print(f'Final Train Loss: {train_losses2[-1]:.3f}')
print(f'Final Val   Loss: {val_losses2[-1]:.3f}')
print(f'Val Perplexity:   {math.exp(val_losses2[-1]):.2f}')

MODEL 2 — emb_dim=768
  Step    0 | Train loss 9.651 | Val loss 9.938
  Step    5 | Train loss 8.102 | Val loss 8.370
Epoch 1 sample:
AI marketing systems can, and, and, and, and, and, and, and, and, and, and, and, and, and, and, and, and, and, and, and, and, and, and, and, and, and

  Step   10 | Train loss 6.582 | Val loss 7.326
Epoch 2 sample:
AI marketing systems can, and, and, and, and, and.                                       

  Step   15 | Train loss 5.514 | Val loss 6.994
Epoch 3 sample:
AI marketing systems can of the system. This. This. This the system of AI. This, and the system the system, and, and, and.                      

  Step   20 | Train loss 4.823 | Val loss 7.036
Epoch 4 sample:
AI marketing systems can data-driven the system to the system that the system of AI-driven, and the system that the most of the most of the most of the most of AI-driven. This and the system to the most, and of the of the system

  Step   25 | Train loss 3.491 | Val loss 7.070
Epoch 5 

In [ ]:
#train model 3
print('=' * 60)
print('MODEL 3 — n_layers=2')
print('=' * 60)
torch.manual_seed(123)
train_loader3 = create_dataloader_v1(train_data, batch_size=2, max_length=124, stride=124, shuffle=True,  drop_last=True)
val_loader3   = create_dataloader_v1(val_data,   batch_size=2, max_length=124, stride=124, shuffle=False, drop_last=False)
model3.to(device)
optimizer3 = torch.optim.AdamW(model3.parameters(), lr=0.0004, weight_decay=0.1)
train_losses3, val_losses3, _ = train_model_simple(
    model3, train_loader3, val_loader3, optimizer3, device,
    num_epochs=10, eval_freq=5, eval_iter=5, start_context=start_context, tokenizer=tokenizer
)
print(f'Final Train Loss: {train_losses3[-1]:.3f}')
print(f'Final Val   Loss: {val_losses3[-1]:.3f}')
print(f'Val Perplexity:   {math.exp(val_losses3[-1]):.2f}')

MODEL 3 — n_layers=2
  Step    0 | Train loss 10.597 | Val loss 10.701
  Step    5 | Train loss 8.396 | Val loss 8.871
Epoch 1 sample:
AI marketing systems can, and, and, and, and, and, and, and, and, and, and, and,,,, and, and,,, and, and., and,,, and,,,, and,, and

  Step   10 | Train loss 6.949 | Val loss 7.727
Epoch 2 sample:
AI marketing systems can. and of of of. of.. and of and. and. and of and of of of and of of. of and. and of. and. and of. of and. of. and of. of of and of. and

  Step   15 | Train loss 5.781 | Val loss 7.035
Epoch 3 sample:
AI marketing systems can, and, and, and, and, and, and, and, and, and, and, and.                           

  Step   20 | Train loss 4.743 | Val loss 6.946
Epoch 4 sample:
AI marketing systems can data. This allows marketers. This. This allows the most of AI. This, and. This allows, and content. This allows the most of the most that reflects. This. This allows, and content. This allows, and content. This

  Step   25 | Train loss 3.889 | 

In [ ]:
#train model 4
print('=' * 60)
print('MODEL 4 — n_layers=12')
print('=' * 60)
torch.manual_seed(123)
train_loader4 = create_dataloader_v1(train_data, batch_size=2, max_length=124, stride=124, shuffle=True,  drop_last=True)
val_loader4   = create_dataloader_v1(val_data,   batch_size=2, max_length=124, stride=124, shuffle=False, drop_last=False)
model4.to(device)
optimizer4 = torch.optim.AdamW(model4.parameters(), lr=0.0004, weight_decay=0.1)
train_losses4, val_losses4, _ = train_model_simple(
    model4, train_loader4, val_loader4, optimizer4, device,
    num_epochs=10, eval_freq=5, eval_iter=5, start_context=start_context, tokenizer=tokenizer
)
print(f'Final Train Loss: {train_losses4[-1]:.3f}')
print(f'Final Val   Loss: {val_losses4[-1]:.3f}')
print(f'Val Perplexity:   {math.exp(val_losses4[-1]):.2f}')

MODEL 4 — n_layers=12
  Step    0 | Train loss 9.651 | Val loss 9.938
  Step    5 | Train loss 8.102 | Val loss 8.370
Epoch 1 sample:
AI marketing systems can, and, and, and, and, and, and, and, and, and, and, and, and, and, and, and, and, and, and, and, and, and, and, and, and, and

  Step   10 | Train loss 6.582 | Val loss 7.326
Epoch 2 sample:
AI marketing systems can, and, and, and, and, and.                                       

  Step   15 | Train loss 5.514 | Val loss 6.994
Epoch 3 sample:
AI marketing systems can of the system. This. This. This the system of AI. This, and the system the system, and, and, and.                      

  Step   20 | Train loss 4.823 | Val loss 7.036
Epoch 4 sample:
AI marketing systems can data-driven the system to the system that the system of AI-driven, and the system that the most of the most of the most of the most of AI-driven. This and the system to the most, and of the of the system

  Step   25 | Train loss 3.491 | Val loss 7.070
Epoch 5 

$$$$$$$$$$$$

In [ ]:
# Load necessary packages
import torch
import math
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import tiktoken
tokenizer = tiktoken.encoding_for_model('gpt-2')

In [ ]:
# Create GPT Data set class
class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []
        token_ids = tokenizer.encode(txt)
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i+max_length]
            target_chunk = token_ids[i+1:i+max_length+1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

In [ ]:
# Create a dataloader function
def create_dataloader_v1(txt, batch_size=4, max_length=124, stride=124, shuffle=True, drop_last=True, num_workers=0):
    tokenizer = tiktoken.get_encoding('gpt2')
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, drop_last=drop_last, num_workers=num_workers)
    return dataloader

In [ ]:
# Create the Multi head attention class
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert (d_out % num_heads == 0), "d_out must be divisible by num_heads"
        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer('mask', torch.triu(torch.ones(context_length, context_length), diagonal=1))

    def forward(self, x):
        b, num_tokens, d_in = x.shape
        keys    = self.W_key(x).view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)
        queries = self.W_query(x).view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)
        values  = self.W_value(x).view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)
        attn_scores = queries @ keys.transpose(2, 3)
        attn_scores.masked_fill_(self.mask.bool()[:num_tokens, :num_tokens], -torch.inf)
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)
        context_vec = (attn_weights @ values).transpose(1, 2).contiguous().view(b, num_tokens, self.d_out)
        return self.out_proj(context_vec)

In [ ]:
# Create the Layer Norm, GELU, and Feed Forward classes
class LayerNorm(nn.Module):
  def __init__(self, emb_dim):
    super().__init__()
    self.eps = 1e-5
    self.scale = nn.Parameter(torch.ones(emb_dim))
    self.shift = nn.Parameter(torch.zeros(emb_dim))

  def forward(self, x):
    mean = x.mean(dim=-1, keepdim=True)
    var  = x.var(dim=-1, keepdim=True, unbiased=False)
    return self.scale * (x - mean) / torch.sqrt(var + self.eps) + self.shift

class GELU(nn.Module):
  def __init__(self):
    super().__init__()

  def forward(self, x):
    return 0.5 * x * (1 + torch.tanh(torch.sqrt(torch.tensor(2.0 / torch.pi)) * (x + 0.044715 * torch.pow(x, 3))))

class FeedForward(nn.Module):
  def __init__(self, cfg):
    super().__init__()
    self.layers = nn.Sequential(
        nn.Linear(cfg['emb_dim'], 4 * cfg['emb_dim']),
        GELU(),
        nn.Linear(4 * cfg['emb_dim'], cfg['emb_dim'])
    )

  def forward(self, x):
    return self.layers(x)


In [ ]:
# Create a Transformer Block class
class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att = MultiHeadAttention(
            d_in=cfg['emb_dim'], d_out=cfg['emb_dim'],
            context_length=cfg['context_length'], num_heads=cfg['n_heads'],
            dropout=cfg['drop_rate'], qkv_bias=cfg['qkv_bias']
        )
        self.ff = FeedForward(cfg)
        self.norm1 = LayerNorm(cfg['emb_dim'])
        self.norm2 = LayerNorm(cfg['emb_dim'])
        self.drop_shortcut = nn.Dropout(cfg['drop_rate'])

    def forward(self, x):
        shortcut = x
        x = self.norm1(x)
        x = self.att(x)
        x = self.drop_shortcut(x)
        x = x + shortcut
        shortcut = x
        x = self.norm2(x)
        x = self.ff(x)
        x = self.drop_shortcut(x)
        return x + shortcut

In [ ]:
# Create the GPT Model class
class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb  = nn.Embedding(cfg['vocab_size'], cfg['emb_dim'])
        self.pos_emb  = nn.Embedding(cfg['context_length'], cfg['emb_dim'])
        self.drop_emb = nn.Dropout(cfg['drop_rate'])
        self.trf_blocks = nn.Sequential(*[TransformerBlock(cfg) for _ in range(cfg['n_layers'])])
        self.final_norm = LayerNorm(cfg['emb_dim'])
        self.out_head   = nn.Linear(cfg['emb_dim'], cfg['vocab_size'], bias=False)

    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape
        x = self.tok_emb(in_idx) + self.pos_emb(torch.arange(seq_len, device=in_idx.device))
        x = self.drop_emb(x)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        return self.out_head(x)

In [ ]:
# Create a function that will generate text
def generate_text_simple(model, idx, max_new_tokens, context_size):
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -context_size:]
        with torch.no_grad():
            logits = model(idx_cond)
        idx_next = torch.argmax(torch.softmax(logits[:, -1, :], dim=-1), dim=-1, keepdim=True)
        idx = torch.cat((idx, idx_next), dim=1)
    return idx

In [ ]:
# Create functions to convert 1) text to tokens and 2) tokens to text
def text_to_token_ids(text, tokenizer):
    encoded = tokenizer.encode(text, allowed_special={'<|endoftext|>'})
    return torch.tensor(encoded).unsqueeze(0)

def token_ids_to_text(token_ids, tokenizer):
    return tokenizer.decode(token_ids.squeeze(0).tolist())

In [ ]:
# Create a function that calculates the cross entropy loss for a single batch
def calc_loss_batch(input_batch, target_batch, model, device):
    input_batch  = input_batch.to(device)
    target_batch = target_batch.to(device)
    logits = model(input_batch)
    return torch.nn.functional.cross_entropy(logits.flatten(0, 1), target_batch.flatten())

In [ ]:
# Create a function that calculates the cross entropy across all batches (the full loader)
def calc_loss_loader(data_loader, model, device, num_batches=None):
    total_loss = 0.
    if len(data_loader) == 0:
        return float('nan')
    num_batches = len(data_loader) if num_batches is None else min(num_batches, len(data_loader))
    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i < num_batches:
            total_loss += calc_loss_batch(input_batch, target_batch, model, device).item()
        else:
            break
    return total_loss / num_batches

In [ ]:
# Create a function that will evaluate your model
def evaluate_model(model, train_loader, val_loader, device, eval_iter):
    model.eval()
    with torch.no_grad():
        train_loss = calc_loss_loader(train_loader, model, device, num_batches=eval_iter)
        val_loss   = calc_loss_loader(val_loader,   model, device, num_batches=eval_iter)
    model.train()
    return train_loss, val_loss

In [ ]:
# Create a function that will print a sample sentence completion
def generate_and_print_sample(model, tokenizer, device, start_context):
    model.eval()
    context_size = model.pos_emb.weight.shape[0]
    encoded = text_to_token_ids(start_context, tokenizer).to(device)
    with torch.no_grad():
        token_ids = generate_text_simple(model=model, idx=encoded, max_new_tokens=50, context_size=context_size)
    print(token_ids_to_text(token_ids, tokenizer).replace('\n', ' '))
    model.train()

In [ ]:
# Create a function for a simple model training loop
def train_model_simple(model, train_loader, val_loader, optimizer, device,
                       num_epochs, eval_freq, eval_iter, start_context, tokenizer):
    train_losses, val_losses, track_tokens_seen = [], [], []
    tokens_seen, global_step = 0, -1
    for epoch in range(num_epochs):
        model.train()
        for input_batch, target_batch in train_loader:
            optimizer.zero_grad()
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            loss.backward()
            optimizer.step()
            tokens_seen += input_batch.numel()
            global_step += 1
            if global_step % eval_freq == 0:
                train_loss, val_loss = evaluate_model(model, train_loader, val_loader, device, eval_iter)
                train_losses.append(train_loss)
                val_losses.append(val_loss)
                track_tokens_seen.append(tokens_seen)
                print(f"  Step {global_step:4d} | Train loss {train_loss:.3f} | Val loss {val_loss:.3f}")
        print(f"Epoch {epoch+1} sample:")
        generate_and_print_sample(model, tokenizer, device, start_context)
        print()
    return train_losses, val_losses, track_tokens_seen

In [ ]:
# Set configurations for model 1
GPT_CONFIG_124M = {
    "vocab_size": 50257, "context_length": 124,
    "emb_dim": 384, "n_heads": 12, "n_layers": 12,
    "drop_rate": 0.1, "qkv_bias": False
}
torch.manual_seed(123)
model1 = GPTModel(GPT_CONFIG_124M)


In [ ]:
# Set configurations for model 2
GPT_CONFIG_124M = {
    "vocab_size": 50257, "context_length": 124,
    "emb_dim": 768, "n_heads": 12, "n_layers": 12,
    "drop_rate": 0.1, "qkv_bias": False
}
torch.manual_seed(123)
model2 = GPTModel(GPT_CONFIG_124M)

In [ ]:
# Set configurations for model 3
GPT_CONFIG_124M = {
    "vocab_size": 50257, "context_length": 124,
    "emb_dim": 768, "n_heads": 12, "n_layers": 2,
    "drop_rate": 0.1, "qkv_bias": False
}
torch.manual_seed(123)
model3 = GPTModel(GPT_CONFIG_124M)

In [ ]:
# Set configurations for model 4
GPT_CONFIG_124M = {
    "vocab_size": 50257, "context_length": 124,
    "emb_dim": 768, "n_heads": 12, "n_layers": 12,
    "drop_rate": 0.1, "qkv_bias": False
}
torch.manual_seed(123)
model4 = GPTModel(GPT_CONFIG_124M)

In [ ]:
# Model 1: Create train / validation data and define train / validation loaders
train_ratio = 0.90
split_idx   = int(train_ratio * len(raw_text))
train_data  = raw_text[:split_idx]
val_data    = raw_text[split_idx:]

torch.manual_seed(123)
train_loader1 = create_dataloader_v1(train_data, batch_size=2, max_length=124, stride=124, shuffle=True,  drop_last=True)
val_loader1   = create_dataloader_v1(val_data,   batch_size=2, max_length=124, stride=124, shuffle=False, drop_last=False)

In [ ]:
# Model 1: Train! (5-10 epochs, make up a start_context that is relevant for your data)
device = 'cpu'
model1.to(device)
optimizer1 = torch.optim.AdamW(model1.parameters(), lr=0.0004, weight_decay=0.1)
train_losses1, val_losses1, _ = train_model_simple(
    model1, train_loader1, val_loader1, optimizer1, device,
    num_epochs=10, eval_freq=5, eval_iter=5,
    start_context='AI marketing systems can', tokenizer=tokenizer
)
print(f"Final Train Loss: {train_losses1[-1]:.3f}")
print(f"Final Val Loss:   {val_losses1[-1]:.3f}")
print(f"Val Perplexity:   {math.exp(val_losses1[-1]):.2f}")

  Step    0 | Train loss 10.303 | Val loss 10.397
  Step    5 | Train loss 9.090 | Val loss 9.173
Epoch 1 sample:
AI marketing systems can,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,

  Step   10 | Train loss 8.161 | Val loss 8.434
Epoch 2 sample:
AI marketing systems can, and, and, and, and, and, and, and, and, and, and, and, and, and, and, and, and, and, and, and, and, and, and, and, and, and

  Step   15 | Train loss 7.096 | Val loss 7.741
Epoch 3 sample:
AI marketing systems can, and, and, and, and, and, and, and, and, and, and, and, and, and, and, and, and, and, and, and, and, and, and and, and, and,

  Step   20 | Train loss 5.941 | Val loss 7.349
Epoch 4 sample:
AI marketing systems can.                                                 

  Step   25 | Train loss 5.181 | Val loss 7.105
Epoch 5 sample:
AI marketing systems can. This allows the and. This. This allows. This allows the, and and, and and and and content, and, and.                      

  Step   30 | Train loss 4

In [ ]:
# Model 2: Create train / validation data and define train / validation loaders
torch.manual_seed(123)
train_loader2 = create_dataloader_v1(train_data, batch_size=2, max_length=124, stride=124, shuffle=True,  drop_last=True)
val_loader2   = create_dataloader_v1(val_data,   batch_size=2, max_length=124, stride=124, shuffle=False, drop_last=False)

In [ ]:
# Model 2: Train! (5-10 epochs, make up a start_context that is relevant for your data)
model2.to(device)
optimizer2 = torch.optim.AdamW(model2.parameters(), lr=0.0004, weight_decay=0.1)
train_losses2, val_losses2, _ = train_model_simple(
    model2, train_loader2, val_loader2, optimizer2, device,
    num_epochs=6, eval_freq=5, eval_iter=5,
    start_context='AI marketing systems can', tokenizer=tokenizer
)
print(f"Final Train Loss: {train_losses2[-1]:.3f}")
print(f"Final Val Loss:   {val_losses2[-1]:.3f}")
print(f"Val Perplexity:   {math.exp(val_losses2[-1]):.2f}")

  Step    0 | Train loss 9.651 | Val loss 9.938
  Step    5 | Train loss 8.102 | Val loss 8.370
Epoch 1 sample:
AI marketing systems can, and, and, and, and, and, and, and, and, and, and, and, and, and, and, and, and, and, and, and, and, and, and, and, and, and

  Step   10 | Train loss 6.582 | Val loss 7.326
Epoch 2 sample:
AI marketing systems can, and, and, and, and, and.                                       

  Step   15 | Train loss 5.514 | Val loss 6.994
Epoch 3 sample:
AI marketing systems can of the system. This. This. This the system of AI. This, and the system the system, and, and, and.                      

  Step   20 | Train loss 4.823 | Val loss 7.036
Epoch 4 sample:
AI marketing systems can data-driven the system to the system that the system of AI-driven, and the system that the most of the most of the most of the most of AI-driven. This and the system to the most, and of the of the system

  Step   25 | Train loss 3.491 | Val loss 7.070
Epoch 5 sample:
AI marketing s

In [ ]:
# Model 3: Create train / validation data and define train / validation loaders
torch.manual_seed(123)
train_loader3 = create_dataloader_v1(train_data, batch_size=2, max_length=124, stride=124, shuffle=True,  drop_last=True)
val_loader3   = create_dataloader_v1(val_data,   batch_size=2, max_length=124, stride=124, shuffle=False, drop_last=False)

In [ ]:
# Model 3: Train! (5-10 epochs, make up a start_context that is relevant for your data)
model3.to(device)
optimizer3 = torch.optim.AdamW(model3.parameters(), lr=0.0004, weight_decay=0.1)
train_losses3, val_losses3, _ = train_model_simple(
    model3, train_loader3, val_loader3, optimizer3, device,
    num_epochs=6, eval_freq=5, eval_iter=5,
    start_context='AI marketing systems can', tokenizer=tokenizer
)
print(f"Final Train Loss: {train_losses3[-1]:.3f}")
print(f"Final Val Loss:   {val_losses3[-1]:.3f}")
print(f"Val Perplexity:   {math.exp(val_losses3[-1]):.2f}")

  Step    0 | Train loss 10.597 | Val loss 10.701
  Step    5 | Train loss 8.396 | Val loss 8.871
Epoch 1 sample:
AI marketing systems can, and, and, and, and, and, and, and, and, and, and, and,,,, and, and,,, and, and., and,,, and,,,, and,, and

  Step   10 | Train loss 6.949 | Val loss 7.727
Epoch 2 sample:
AI marketing systems can. and of of of. of.. and of and. and. and of and of of of and of of. of and. and of. and. and of. of and. of. and of. of of and of. and

  Step   15 | Train loss 5.781 | Val loss 7.035
Epoch 3 sample:
AI marketing systems can, and, and, and, and, and, and, and, and, and, and, and.                           

  Step   20 | Train loss 4.743 | Val loss 6.946
Epoch 4 sample:
AI marketing systems can data. This allows marketers. This. This allows the most of AI. This, and. This allows, and content. This allows the most of the most that reflects. This. This allows, and content. This allows, and content. This

  Step   25 | Train loss 3.889 | Val loss 6.905
Epoch 

In [ ]:
# Model 4: Create train / validation data and define train / validation loaders
torch.manual_seed(123)
train_loader4 = create_dataloader_v1(train_data, batch_size=2, max_length=124, stride=124, shuffle=True,  drop_last=True)
val_loader4   = create_dataloader_v1(val_data,   batch_size=2, max_length=124, stride=124, shuffle=False, drop_last=False)

In [ ]:
# Model 4: Train! (5-10 epochs, make up a start_context that is relevant for your data)
model4.to(device)
optimizer4 = torch.optim.AdamW(model4.parameters(), lr=0.0004, weight_decay=0.1)
train_losses4, val_losses4, _ = train_model_simple(
    model4, train_loader4, val_loader4, optimizer4, device,
    num_epochs=6, eval_freq=5, eval_iter=5,
    start_context='AI marketing systems can', tokenizer=tokenizer
)
print(f"Final Train Loss: {train_losses4[-1]:.3f}")
print(f"Final Val Loss:   {val_losses4[-1]:.3f}")
print(f"Val Perplexity:   {math.exp(val_losses4[-1]):.2f}")

  Step    0 | Train loss 9.651 | Val loss 9.938
  Step    5 | Train loss 8.102 | Val loss 8.370
Epoch 1 sample:
AI marketing systems can, and, and, and, and, and, and, and, and, and, and, and, and, and, and, and, and, and, and, and, and, and, and, and, and, and

  Step   10 | Train loss 6.582 | Val loss 7.326
Epoch 2 sample:
AI marketing systems can, and, and, and, and, and.                                       

  Step   15 | Train loss 5.514 | Val loss 6.994
Epoch 3 sample:
AI marketing systems can of the system. This. This. This the system of AI. This, and the system the system, and, and, and.                      

  Step   20 | Train loss 4.823 | Val loss 7.036
Epoch 4 sample:
AI marketing systems can data-driven the system to the system that the system of AI-driven, and the system that the most of the most of the most of the most of AI-driven. This and the system to the most, and of the of the system

  Step   25 | Train loss 3.491 | Val loss 7.070
Epoch 5 sample:
AI marketing s

**RESULTS**

1. Model 1
    + Train Loss: 1.775
    + Val Loss: 6.896
    + Val Perplexity: 988.36
    + Final Sentence Competion: AI markleting systems can identify the system that evolves over time...

2. Model 2
    + Train Loss: 2.027
    + Val Loss: 7.121
    + Val Perplexity: 1237.73
    + Final Sentence Competion: AI marketing systems can are is anothjer systems thinker who understands how algorithms influence...

3. Model 3
    + Train Loss: 0.687
    + Val Loss: 7.107
    + Val Perplexity: 1220.94
    + Final Sentence Competion: AI marketing systems can identify new the way businesses approach...

4. Model 4
    + Train Loss: 0.325
    + Val Loss: 7.255
    + Val Perplexity: 1415.86
    + Final Sentence Competion: AI marketing systems can identify new users now drive billions olf dollars in revenue...


_______________________________________________
Which model did best? Did Model 1 vs Model 2 and Model 3 vs Model 4 perform as expected (better or worse based on settings)? Which appeared to have the bigger impact, the parameter you changed between Models 1 & 2 or the parameter you changed between Models 3 & 4?
_______________________________________________

Model 1 vs Model 2 (embedding size)
It was expected that Model 2 (larger embedding = 768) would perform better than Model 1 (384). However, Model 1 actually performed better on validation metrics, meaning the larger embedding size led to worse generalization, likely due to overfitting on the small dataset.
Model 3 vs Model 4 (number of layers)
It was expected that Model 4 (12 layers) would outperform Model 3 (2 layers). However, Model 3 slightly outperformed Model 4 on validation loss, again suggesting that the deeper model overfit and did not generalize as well.

The number of transformer layers (depth) appeared to have the bigger impact.

The gap between Model 3 and Model 4 showed that increasing depth significantly changed both training behavior and generalization (lower train loss but worse validation).
While embedding size also mattered, the difference between Model 1 and Model 2 was less dramatic in terms of validation performance.


$$$$
Model 1 performed the best overall based on validation loss and perplexity. The results did not fully match expectations, as increasing embedding dimension (Model 2) and increasing depth (Model 4) both led to worse validation performance rather than improvement. This indicates that the larger models overfit the relatively small dataset. Between the two parameters tested, increasing the number of transformer layers had a greater impact on performance than increasing embedding dimension, as it more strongly affected both training loss and generalization behavior.